In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import json
import brightway2 as bw
from brightway2 import *
import bw2data

# import own Python files, vars, mappings, and functions
from config import (CC_METHOD, NAME_REF_DB, NAME_FUTURE_DB,
             PROJECT_NAME, OUT_JSON_GHG, OUT_JSON_GHG_FUTURE, 
                    COST_DATA_PTX, OUT_JSON_GHG_PTX, OUT_JSON_GHG_PTX_FUTURE)

import create_db_lca_functions as lcaf


In [ ]:
GENERATE_NEW_LCA_DB=True
CALC_ALL_LCA_IMPACTS=True

In [ ]:
PROJECT_NAME

import brightway2 as bw
if 'biosphere3' in list(bw.databases):
    print("Deleting existing database...")
    del bw.databases['biosphere3']
bw.databases

In [ ]:
bw2data.projects.set_current(PROJECT_NAME)

# If we want to use a full LCA approach, we have to set-up the LCA database:
if GENERATE_NEW_LCA_DB:
    bw2data.projects.set_current(PROJECT_NAME)
    # Import LCIA methods, import ecoinvent cut-off and consequential dbs
    #lcaf.import_additional_lcias()
    lcaf.import_ecoinvent_database()
    # Generate reference database with premise to import additional novel LCIs.
    lcaf.generate_reference_database()

    # Make future scenario, use 2C scenario from REMIND
    list_spec_scenarios, list_names = lcaf.generate_future_ei_dbs(
        scenarios=["SSP2-PkBudg1150"],
        iam="remind",
        start_yr=2025,
        end_yr=2050,
        step=25,
        endstring="base",
    )
    lcaf.generate_prospective_lca_dbs(list_spec_scenarios, list_names)

# Generate PtX GHG emission data
if CALC_ALL_LCA_IMPACTS:
    if COST_DATA_PTX is None:
        raise ValueError("COST_DATA_PTX is None; cannot export PtX GHG factors")
    cost_dict_ptx = COST_DATA_PTX[NAME_REF_DB].to_dict()
    dict_ghg_impacts_ptx = lcaf.get_tech_environmental_burdens_ptx(cost_dict_ptx, sec_db=NAME_REF_DB)
    with open(OUT_JSON_GHG_PTX, "w") as file:
        json.dump(dict_ghg_impacts_ptx, file)

    cost_dict_ptx_future = COST_DATA_PTX[NAME_FUTURE_DB].to_dict()
    dict_ghg_impacts_ptx_future = lcaf.get_tech_environmental_burdens_ptx(
        cost_dict_ptx_future, sec_db=NAME_FUTURE_DB
    )
    with open(OUT_JSON_GHG_PTX_FUTURE, "w") as file:
        json.dump(dict_ghg_impacts_ptx_future, file)
else:
    with open(OUT_JSON_GHG_PTX, "r") as file:
        dict_ghg_impacts_ptx = json.load(file)
    with open(OUT_JSON_GHG_PTX_FUTURE, "r") as file:
        dict_ghg_impacts_ptx_future = json.load(file)

dict_ghg_impacts_ptx


In [ ]:
import create_db_lca_functions as lcaf
import gc
from config import NAME_FUTURE_DB

# Check if it already exists
import brightway2 as bw
if NAME_FUTURE_DB in bw.databases:
    print(f"Future database already exists, deleting to regenerate...")
    del bw.databases[NAME_FUTURE_DB]

# Generate future database ONLY
print("Generating future database...")
list_spec_scenarios, list_names = lcaf.generate_future_ei_dbs(
    scenarios=["SSP2-PkBudg1150"], 
    iam='remind',
    start_yr=2025, 
    end_yr=2050, 
    step=25, 
    endstring="base"
)

print(f"Scenarios to generate: {list_names}")

lcaf.generate_prospective_lca_dbs(list_spec_scenarios, list_names)

# Clean memory
gc.collect()

print(f"✓ Future database generated: {NAME_FUTURE_DB}")

# Verify
if NAME_FUTURE_DB in bw.databases:
    db = bw.Database(NAME_FUTURE_DB)
    print(f"  Database has {len(db)} activities")
    
    # Check for PtX
    ptx = [a for a in db if 'methanol production, infrastructure' in a['name'].lower()]
    print(f"  Found {len(ptx)} PtX infrastructure activities")
else:
    print("  ❌ Database was not created")

In [ ]:
import brightway2 as bw
from config import NAME_REF_DB, CC_METHOD

db = bw.Database(NAME_REF_DB)

# Get the activity
act = [a for a in db if a['name'] == 'carbon dioxide production, infrastructure and catalyst' 
       and a['location'] == 'GLO'][0]

print(f"Activity: {act['name']}")
print(f"Reference product: {act.get('reference product')}")
print(f"Unit: {act.get('unit')}")
print(f"\nExchanges:")

# Do LCA
lca = bw.LCA({act: 1}, method=CC_METHOD)
lca.lci()
lca.lcia()

print(f"\nTotal impact: {lca.score}")

# Check individual exchanges
for exc in act.exchanges():
    if exc['type'] == 'technosphere':
        lca.redo_lcia({exc.input: exc['amount']})
        if abs(lca.score) > 0.01:  # Only show significant contributions
            print(f"  Tech: {exc.input['name'][:50]:50s} | {exc['amount']:10.2e} | Impact: {lca.score:10.4f}")
    elif exc['type'] == 'biosphere':
        cf = lca.characterization_matrix[lca.biosphere_dict[exc.input], :].sum()
        impact = cf * exc['amount']
        if abs(impact) > 0.01:
            print(f"  Bio:  {exc.input['name'][:50]:50s} | {exc['amount']:10.2e} | Impact: {impact:10.4f}")

In [ ]:
dict_ghg_impacts_ptx_future

## Generate pre-calculated GHG emission factors for the grid

In [ ]:
metadata_list = []
for db in [NAME_REF_DB, NAME_FUTURE_DB]:
    all_acts = [act for act in bw.Database(db) if ('market for electricity, low voltage' == act['name'] or 'market group for electricity, low voltage' == act['name']) and 'electricity, low voltage' == act['reference product'] ] 
    
    for act_sel in all_acts:
        """Store metadata"""
        metadata = {
            'location': act_sel.get('location', ''),
            "db": db,
            'name': act_sel.get('name', ''),
            'unit': act_sel.get('unit', ''),
            'reference_product': act_sel.get('reference product', ''),
            'key': act_sel.key,
            
        }
        metadata_list.append(metadata)

df_meta = pd.DataFrame(metadata_list)
df_meta

In [ ]:
def run_mlca(
    activity_keys: list,
    functional_units: list,
    result_index_labels: list,
    column_suffix: str,
    impact_methods: list,
) -> pd.DataFrame:
    """
    Run a MultiLCA calculation and return a DataFrame of results.

    Parameters:
    - activity_keys: list of Brightway activity keys for LCA calculation
    - functional_units: list of float values (same length as activity_keys), quantities of each key
    - result_index_labels: labels for resulting DataFrame index (must match order of keys)
    - column_suffix: suffix for LCA impact columns (e.g., '', '_ref')
    - impact_methods: list of LCIA methods to evaluate

    Returns:
    - pd.DataFrame: LCA results with impacts per row (indexed by result_index_labels)
    """
    assert len(activity_keys) == len(functional_units) == len(result_index_labels), \
        "Keys, functional units, and labels must be the same length."

    setup_name = f"mlca_setup_{column_suffix.strip('_') or 'main'}"

    # Prepare inventory with variable functional units
    inventory = [{key: fu} for key, fu in zip(activity_keys, functional_units)]

    # Define calculation setup
    bw.calculation_setups[setup_name] = {
        'inv': inventory,
        'ia': impact_methods,
        'name': setup_name,
    }

    # Run MultiLCA
    mlca = MultiLCA(setup_name)
    results_array = mlca.results

    # Create labeled results DataFrame
    result_df = pd.DataFrame(
        data=results_array,
        index=result_index_labels,
        columns=[f"lca_impact{column_suffix}_{method[1]}" for method in impact_methods]
    )
    #result_df['key'] = activity_keys

    return result_df

In [ ]:
CC_METHOD

In [ ]:
df_lca = run_mlca(
    activity_keys=df_meta.key,
    functional_units= [1] * len(df_meta),
    result_index_labels=df_meta.key,
    column_suffix="",
    impact_methods=[CC_METHOD]
)

df_total_power = df_meta.merge(df_lca, left_on=['key'], right_index=True, how='left')
df_total_power

In [ ]:
df_total_power_2025=df_total_power[df_total_power['db'] == NAME_REF_DB]
df_total_power_2050=df_total_power[df_total_power['db'] == NAME_FUTURE_DB]

# Convert to dictionary indexed by 'location'
total_power_dict = df_total_power_2025.set_index('location').to_dict(orient='index')
total_power_dict_future = df_total_power_2050.set_index('location').to_dict(orient='index')

# Save to JSON file
with open(OUT_JSON_GHG, "w") as f:
    json.dump(total_power_dict, f, indent=2)

with open(OUT_JSON_GHG_FUTURE, "w") as f:
    json.dump(total_power_dict_future, f, indent=2)